Pull WRDS Data

In [12]:
import wrds
import pandas as pd
import numpy as np

db = wrds.Connection(wrds_username="skurono")
db.create_pgpass_file()
print("Connected to WRDS")

Loading library list...
Done
Connected to WRDS


Check data schemas

In [ ]:
print(db.describe_table(library='crsp', table='msf'))

# make sure hexcd is exhcange codes
print(db.raw_sql("SELECT hexcd, COUNT(*) FROM crsp.msf GROUP BY hexcd ORDER BY hexcd"))

print(db.list_tables(library="crsp"))



Approximately 5153763 rows in crsp.msf.
        name  nullable              type  \
0      cusip      True        VARCHAR(8)   
1     permno      True           INTEGER   
2     permco      True           INTEGER   
3     issuno      True           INTEGER   
4      hexcd      True          SMALLINT   
5     hsiccd      True           INTEGER   
6       date      True              DATE   
7      bidlo      True    NUMERIC(11, 5)   
8      askhi      True    NUMERIC(11, 5)   
9        prc      True    NUMERIC(11, 5)   
10       vol      True    NUMERIC(10, 0)   
11       ret      True    NUMERIC(10, 6)   
12       bid      True    NUMERIC(11, 5)   
13       ask      True    NUMERIC(11, 5)   
14    shrout      True  DOUBLE PRECISION   
15    cfacpr      True  DOUBLE PRECISION   
16   cfacshr      True  DOUBLE PRECISION   
17    altprc      True    NUMERIC(11, 5)   
18    spread      True    NUMERIC(10, 5)   
19  altprcdt      True              DATE   
20      retx      True    NUMERIC(10

Table 1: Pull constituents list from wrds

In [13]:
data = db.raw_sql("""SELECT permno, start, ending
                     FROM crsp.msp500list""")

data["start"]  = pd.to_datetime(data["start"])
data["ending"] = pd.to_datetime(data["ending"])

#verify data
print(data.shape)
print(data.head())
print(data["start"].dtype)
print(data["ending"].dtype)
print(data.duplicated(subset=['permno','start','ending']).sum())

data.to_parquet("data_wrds/constituents.parquet")

(2064, 3)
   permno      start     ending
0   10006 1957-03-01 1984-07-18
1   10030 1957-03-01 1969-01-08
2   10049 1925-12-31 1932-10-01
3   10057 1957-03-01 1992-07-02
4   10078 1992-08-20 2010-01-28
datetime64[ns]
datetime64[ns]
0


Table 2: Pull monthly returns for constituents list

In [22]:
permnos = tuple(data["permno"].unique().tolist())
print(len(permnos))

monthly_returns_all_constituents_df = db.raw_sql(f"""SELECT permno, date, ret, retx, prc, shrout, hexcd
                                                    FROM crsp.msf
                                                    WHERE hexcd IN (1,2,3)
                                                      AND date >= '1989-01-01'
                                                      AND permno IN {permnos}
                                                """)

1936


In [ ]:

# change type from string to datetime
monthly_returns_all_constituents_df["date"] = pd.to_datetime(monthly_returns_all_constituents_df["date"])

# check shape
print(monthly_returns_all_constituents_df.shape)
print(monthly_returns_all_constituents_df.head())

# check dtypes
print(monthly_returns_all_constituents_df.dtypes)

# Check NAs
print(monthly_returns_all_constituents_df.isna().sum())

# Save to parquet
monthly_returns_all_constituents_df.to_parquet("data_wrds/monthly_returns_all_constituents.parquet")

(352196, 7)
   permno       date       ret      retx     prc  shrout  hexcd
0   10057 1989-01-31  0.063158  0.052632    10.0  6280.0      1
1   10057 1989-02-28    0.0375    0.0375  10.375  6280.0      1
2   10057 1989-03-31  0.012048  0.012048    10.5  6280.0      1
3   10057 1989-04-28  -0.02381  -0.02381   10.25  6280.0      1
4   10057 1989-05-31  0.046342  0.036585  10.625  6280.0      1
permno             Int64
date      datetime64[ns]
ret              Float64
retx             Float64
prc              Float64
shrout           Float64
hexcd              Int64
dtype: object
permno       0
date         0
ret       2327
retx      2327
prc       1759
shrout     514
hexcd        0
dtype: int64


Table 3: Pull delisting returns

In [ ]:
delistings_df = db.raw_sql(f"""SELECT permno, dlstdt, dlret, dlstcd
                                FROM crsp.msedelist
                                WHERE permno IN {permnos}
                            """)

In [33]:
# clean up date format
delistings_df["dlstdt"] = pd.to_datetime(delistings_df["dlstdt"])

# validate dataset shape
print(delistings_df.shape)
print(delistings_df.head())

# check dtypes
print(delistings_df.dtypes)

# check nas
print(delistings_df.isna().sum())

delistings_df.to_parquet("data_wrds/delistings.parquet")

(1936, 4)
   permno     dlstdt     dlret  dlstcd
0   10006 1984-06-28   0.03563     233
1   10030 1968-12-26     0.168     200
2   10049 1932-09-28 -0.647059     500
3   10057 1996-07-02       0.0     233
4   10078 2010-01-26  0.013874     233
permno             Int64
dlstdt    datetime64[ns]
dlret            Float64
dlstcd             Int64
dtype: object
permno      0
dlstdt      0
dlret     668
dlstcd      0
dtype: int64


Build point-in-time return panel

Step 1. Build point in time membership panel from table 1